# CF4 — Telegraph–Fisher Causality

- Canon (anchor-only; do not duplicate): [../../Complete-Formalisms/CF4_Telegraph_Fisher_Causality.md](../../Complete-Formalisms/CF4_Telegraph_Fisher_Causality.md)
- Purpose: step-by-step, runnable walkthrough that demonstrates a falsifiable contrast:
  1) Fisher–KPP (RD) front-speed check vs theory.
  2) Telegraph (hyperbolic) finite-speed propagation check against light-cone speed c.

Navigation anchors (canon registries):
- Scale/causality program (A6 notes and RD vs hyperbolic split): see AXIOMS and linked equations.
- Metriplectic/hyperbolic limb (local causality evidence in KG-like transport): see canon.


## Part A — Fisher–KPP front speed (RD)

We simulate 1D Fisher–KPP: $\partial_t u = D\,\partial_{xx}u + r\,u(1-u)$ with explicit Euler (stable for small $\Delta t \le \mathcal O(\Delta x^2/D)$). For small seeds, the pulled-front theory predicts speed $v_\text{pred} = 2\sqrt{Dr}$ (dimensionally reduced units).

Falsifiable check: measure front speed $v_\text{meas}$ by threshold-crossing, compute relative error vs $2\sqrt{Dr}$. Not a benchmark—just enough to detect gross errors.

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

# Parameters (dimensionless)
N = 1024
L = 200.0
x = np.linspace(0.0, L, N, endpoint=False)
dx = x[1] - x[0]
D = 1.0
r = 0.25

# Stability-oriented time step (crude): dt <= dx^2/(2D)
dt = 0.2 * dx*dx / D
steps = 8000
thr = 0.5

# Initial condition: small seed near x=0
u = np.zeros_like(x)
u[x < 5*dx] = 1e-3

def laplacian_periodic(u):
    return (np.roll(u, -1) - 2*u + np.roll(u, 1)) / (dx*dx)

def step_rd(u):
    return u + dt * (D * laplacian_periodic(u) + r * u * (1.0 - u))

def front_pos(u, thr=0.5):
    idx = np.where(u >= thr)[0]
    return float(x[idx[0]]) if len(idx) > 0 else 0.0

p0 = front_pos(u, thr)
for n in range(steps):
    u = step_rd(u)
p1 = front_pos(u, thr)
v_meas = (p1 - p0) / (steps * dt)
v_pred = 2.0 * np.sqrt(D * r)
rel_err = abs(v_meas - v_pred) / max(1e-12, v_pred)
print({'v_meas': v_meas, 'v_pred': v_pred, 'relative_error': rel_err})


### Interpretation
- If the implementation is sane, `relative_error` should be modest (coarse grid/time step bias exists), indicating a plausible Fisher–KPP front realization.
- Large errors indicate instability, thresholding pathology, or discretization bugs (falsifiable).

## Part B — Telegraph finite-speed causality

We simulate the damped wave (telegraph) equation: $u_{tt} + a\,u_t = c^2 u_{xx}$.
Set $v = u_t$ to obtain first-order system $\begin{cases} u_t = v \\ v_t = c^2 u_{xx} - a v \end{cases}$.
Use a leapfrog-like explicit scheme under CFL: $c\,\Delta t/\Delta x \le 1$.

Falsifiable check: compute the radius $R(t)$ where $|u| \ge \theta$ (small threshold) and verify that the apparent propagation speed does not exceed $c$ by more than a small tolerance.

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

# Grid and params
N = 2048
L = 400.0
x = np.linspace(-L/2, L/2, N, endpoint=False)
dx = x[1] - x[0]
c = 1.0
a = 0.05  # damping

# CFL for wave-like transport
dt = 0.9 * dx / max(1e-12, c)
T = 2000
theta = 1e-3

# Initial Gaussian pulse
u = np.exp(- (x/2.0)**2)
v = np.zeros_like(u)

def lap(u):
    return (np.roll(u, -1) - 2*u + np.roll(u, 1)) / (dx*dx)

def step_telegraph(u, v):
    # u_t = v
    # v_t = c^2 u_xx - a v
    u_new = u + dt * v
    v_new = v + dt * (c*c * lap(u) - a * v)
    return u_new, v_new

def support_radius(u, theta):
    idx = np.where(np.abs(u) >= theta)[0]
    if len(idx) == 0:
        return 0.0
    return max(abs(x[idx[0]]), abs(x[idx[-1]]))

R0 = support_radius(u, theta)
Rmax = R0
for n in range(1, T+1):
    u, v = step_telegraph(u, v)
    R = support_radius(u, theta)
    Rmax = max(Rmax, R)

t = T * dt
v_eff = (Rmax - R0) / max(1e-12, t)
print({'R0': R0, 'Rmax': Rmax, 't': t, 'v_eff': v_eff, 'c': c, 'ratio_v_eff_over_c': v_eff/max(1e-12, c)})


### Interpretation
- Expect `ratio_v_eff_over_c` ≲ 1 (within discretization tolerance) for the telegraph scheme under CFL. If it significantly exceeds 1, the scheme or thresholding is suspect (falsifiable).
- Unlike RD, telegraph exhibits a finite effective speed tied to c.

## Summary — Minimal, falsifiable causality contrast

- RD (Fisher–KPP): measured front speed compared to $2\sqrt{Dr}$.
- Telegraph: effective propagation speed compared to c.

Both checks run in a few seconds and are deterministic NumPy; they provide immediate sanity for the CF4 formalism without invoking the full proposal pipeline.

## Repro notes

- Determinism: pure NumPy; IEEE-754 double precision assumed.
- No files are written; production gates/figures must route via `io_paths` per repository policy.

In [ ]:
# io_paths bootstrap (optional, no file writes in this notebook)
from pathlib import Path
import sys
COMMON = Path.cwd().resolve() / 'Derivation' / 'code' / 'common'
if COMMON.exists() and str(COMMON) not in sys.path:
    sys.path.insert(0, str(COMMON))
try:
    from io_paths import figure_path, log_path  # noqa: F401
except Exception as e:
    print('[warn] io_paths not available:', e)
